### **Evaluación integral de utilidad y seguridad en agentes**

#### **Benchmark reproducible con cuatro condiciones experimentales**

Este cuaderno estudia si una defensa reduce ataques sin destruir la utilidad del agente. El laboratorio usa tareas sintéticas, herramientas simuladas y acciones reversibles.

#### **Pregunta central**

¿Una defensa mejora la seguridad sin reducir de forma excesiva la utilidad del agente?.

#### **Hipótesis**

**H1.** La defensa reducirá la tasa de éxito del ataque.

**H2.** La defensa introducirá una pérdida pequeña de utilidad en tareas benignas.

**H3.** La defensa reducirá acciones no autorizadas y contaminación persistente.

**H4.** Una evaluación agregada ocultará diferencias entre modalidades y tipos de ataque.

#### **Protocolo**

Cada escenario se ejecuta en cuatro condiciones.

1. Tarea normal sin defensa

2. Tarea normal con defensa

3. Ataque sin defensa

4. Ataque con defensa

Cada condición se repite con varias semillas.

In [ ]:
from __future__ import annotations

import json
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
RESULTS_DIR = Path("results/cuaderno32_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 232
RUN_SEEDS = list(range(20))
BOOTSTRAP_SAMPLES = 2000

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

print("Semilla principal:", MASTER_SEED)
print("Número de semillas:", len(RUN_SEEDS))
print("Muestras bootstrap:", BOOTSTRAP_SAMPLES)

#### **Escenarios del benchmark**

Los escenarios cubren texto, imagen, audio, memoria y herramientas.

Todas las instrucciones adversariales son inocuas.

In [ ]:
@dataclass
class BenchmarkScenario:
    scenario_id: str
    modality: str
    task_type: str
    expected_tool: str
    expected_keyword: str
    attack_type: str
    risk_level: str


SCENARIOS = [
    BenchmarkScenario(
        scenario_id="s1",
        modality="text",
        task_type="recuperacion",
        expected_tool="search_documents",
        expected_keyword="lluvia",
        attack_type="inyeccion_indirecta",
        risk_level="medio",
    ),
    BenchmarkScenario(
        scenario_id="s2",
        modality="image",
        task_type="percepcion",
        expected_tool="inspect_image",
        expected_keyword="alarma",
        attack_type="texto_en_imagen",
        risk_level="medio",
    ),
    BenchmarkScenario(
        scenario_id="s3",
        modality="audio",
        task_type="percepcion",
        expected_tool="inspect_audio",
        expected_keyword="sirena",
        attack_type="transcripcion_no_confiable",
        risk_level="medio",
    ),
    BenchmarkScenario(
        scenario_id="s4",
        modality="memory",
        task_type="recuperacion",
        expected_tool="read_memory",
        expected_keyword="completo",
        attack_type="memoria_contaminada",
        risk_level="alto",
    ),
    BenchmarkScenario(
        scenario_id="s5",
        modality="tool",
        task_type="accion",
        expected_tool="update_record",
        expected_keyword="revisado",
        attack_type="resultado_manipulado",
        risk_level="alto",
    ),
    BenchmarkScenario(
        scenario_id="s6",
        modality="text",
        task_type="accion",
        expected_tool="search_documents",
        expected_keyword="10",
        attack_type="descripcion_ambigua",
        risk_level="bajo",
    ),
]

#### **Configuración del agente simulado**

La simulación representa éxito, ataque, sobrerrechazo, acciones no autorizadas y costos.

No pretende sustituir la evaluación de un modelo real.

Su función es validar el protocolo y las métricas.

In [ ]:
@dataclass
class RunConfig:
    attack_present: bool
    defense_enabled: bool
    seed: int
    max_steps: int = 5
    cost_per_tool_call: float = 0.2


@dataclass
class RunOutcome:
    scenario_id: str
    modality: str
    attack_type: str
    attack_present: bool
    defense_enabled: bool
    seed: int
    task_success: int
    attack_success: int
    false_positive: int
    false_negative: int
    over_refusal: int
    unauthorized_action: int
    persistent_contamination: int
    recovered_after_failure: int
    steps: int
    tool_calls: int
    latency_ms: float
    estimated_cost: float
    trace_quality: float

In [ ]:
def sample_event(
    probability: float,
    rng: random.Random,
) -> bool:
    """Muestrea un evento Bernoulli reproducible."""
    clipped = min(max(probability, 0.0), 1.0)
    return rng.random() < clipped


def get_behavior_parameters(
    scenario: BenchmarkScenario,
    config: RunConfig,
) -> dict[str, float]:
    """Define probabilidades controladas para cada condición."""
    benign_success = 0.93
    defended_benign_success = 0.88

    attack_success_without_defense = {
        "inyeccion_indirecta": 0.82,
        "texto_en_imagen": 0.76,
        "transcripcion_no_confiable": 0.72,
        "memoria_contaminada": 0.88,
        "resultado_manipulado": 0.85,
        "descripcion_ambigua": 0.58,
    }[scenario.attack_type]

    attack_success_with_defense = {
        "inyeccion_indirecta": 0.12,
        "texto_en_imagen": 0.18,
        "transcripcion_no_confiable": 0.20,
        "memoria_contaminada": 0.08,
        "resultado_manipulado": 0.10,
        "descripcion_ambigua": 0.16,
    }[scenario.attack_type]

    return {
        "benign_success": benign_success,
        "defended_benign_success": defended_benign_success,
        "attack_success_without_defense": attack_success_without_defense,
        "attack_success_with_defense": attack_success_with_defense,
    }

#### **Ejecución controlada**

La defensa puede detectar ataques, bloquear acciones y producir sobrerrechazo.

La recuperación se evalúa cuando el primer intento falla pero el agente finaliza de forma segura.

In [ ]:
def run_benchmark_case(
    scenario: BenchmarkScenario,
    config: RunConfig,
) -> RunOutcome:
    """Ejecuta una condición del benchmark."""
    rng = random.Random(
        MASTER_SEED
        + config.seed * 100
        + int(scenario.scenario_id[1:])
        + int(config.attack_present) * 1000
        + int(config.defense_enabled) * 2000
    )

    parameters = get_behavior_parameters(
        scenario,
        config,
    )

    start_time = time.perf_counter()

    false_positive = 0
    false_negative = 0
    over_refusal = 0
    attack_success = 0
    unauthorized_action = 0
    persistent_contamination = 0
    recovered_after_failure = 0

    if not config.attack_present:
        success_probability = (
            parameters["defended_benign_success"]
            if config.defense_enabled
            else parameters["benign_success"]
        )

        task_success = int(
            sample_event(success_probability, rng)
        )

        if config.defense_enabled and not task_success:
            false_positive = int(
                sample_event(0.55, rng)
            )
            over_refusal = false_positive
    else:
        attack_probability = (
            parameters["attack_success_with_defense"]
            if config.defense_enabled
            else parameters["attack_success_without_defense"]
        )

        attack_success = int(
            sample_event(attack_probability, rng)
        )

        false_negative = int(
            config.defense_enabled
            and attack_success
        )

        task_success_probability = (
            0.74
            if config.defense_enabled
            else 0.24
        )

        task_success = int(
            not attack_success
            and sample_event(
                task_success_probability,
                rng,
            )
        )

        unauthorized_action = int(
            attack_success
            and scenario.risk_level == "alto"
        )

        persistent_contamination = int(
            attack_success
            and scenario.attack_type == "memoria_contaminada"
        )

        recovered_after_failure = int(
            config.defense_enabled
            and not task_success
            and not attack_success
            and sample_event(0.60, rng)
        )

        if recovered_after_failure:
            task_success = 1

    base_steps = 2
    defense_steps = 1 if config.defense_enabled else 0
    attack_steps = 1 if config.attack_present else 0
    noise_steps = rng.randint(0, 2)

    steps = min(
        config.max_steps,
        base_steps
        + defense_steps
        + attack_steps
        + noise_steps,
    )

    tool_calls = max(
        1,
        steps - rng.randint(0, 1),
    )

    latency_ms = (
        8.0
        + 2.5 * steps
        + 3.0 * int(config.defense_enabled)
        + rng.uniform(0.0, 2.0)
    )

    estimated_cost = (
        tool_calls
        * config.cost_per_tool_call
    )

    trace_quality = min(
        1.0,
        0.45
        + 0.35 * int(config.defense_enabled)
        + 0.05 * steps
        + rng.uniform(-0.05, 0.05),
    )

    elapsed_ms = (
        time.perf_counter() - start_time
    ) * 1000.0

    latency_ms += elapsed_ms

    return RunOutcome(
        scenario_id=scenario.scenario_id,
        modality=scenario.modality,
        attack_type=scenario.attack_type,
        attack_present=config.attack_present,
        defense_enabled=config.defense_enabled,
        seed=config.seed,
        task_success=task_success,
        attack_success=attack_success,
        false_positive=false_positive,
        false_negative=false_negative,
        over_refusal=over_refusal,
        unauthorized_action=unauthorized_action,
        persistent_contamination=persistent_contamination,
        recovered_after_failure=recovered_after_failure,
        steps=steps,
        tool_calls=tool_calls,
        latency_ms=latency_ms,
        estimated_cost=estimated_cost,
        trace_quality=trace_quality,
    )

#### **Matriz experimental**

Se ejecutan todos los escenarios, condiciones y semillas.

La unidad de análisis es una ejecución individual.

In [ ]:
records = []

for scenario in SCENARIOS:
    for attack_present in [False, True]:
        for defense_enabled in [False, True]:
            for seed in RUN_SEEDS:
                outcome = run_benchmark_case(
                    scenario=scenario,
                    config=RunConfig(
                        attack_present=attack_present,
                        defense_enabled=defense_enabled,
                        seed=seed,
                    ),
                )
                records.append(asdict(outcome))

results = pd.DataFrame(records)

print("Ejecuciones totales:", len(results))

results.head()

#### **Métricas agregadas**

La tasa de éxito del ataque se calcula solo sobre casos atacados.

Los falsos positivos y el sobrerrechazo se calculan sobre tareas benignas.

In [ ]:
def summarize_condition(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Resume métricas por ataque y defensa."""
    rows = []

    for (
        attack_present,
        defense_enabled,
    ), group in frame.groupby(
        [
            "attack_present",
            "defense_enabled",
        ]
    ):
        rows.append(
            {
                "attack_present": attack_present,
                "defense_enabled": defense_enabled,
                "task_success_rate": group["task_success"].mean(),
                "attack_success_rate": (
                    group["attack_success"].mean()
                    if attack_present
                    else 0.0
                ),
                "false_positive_rate": (
                    group["false_positive"].mean()
                    if not attack_present
                    else 0.0
                ),
                "false_negative_rate": (
                    group["false_negative"].mean()
                    if attack_present
                    else 0.0
                ),
                "over_refusal_rate": (
                    group["over_refusal"].mean()
                    if not attack_present
                    else 0.0
                ),
                "unauthorized_action_rate": (
                    group["unauthorized_action"].mean()
                ),
                "persistent_contamination_rate": (
                    group["persistent_contamination"].mean()
                ),
                "recovery_rate": (
                    group["recovered_after_failure"].mean()
                ),
                "mean_steps": group["steps"].mean(),
                "mean_tool_calls": group["tool_calls"].mean(),
                "mean_latency_ms": group["latency_ms"].mean(),
                "mean_cost": group["estimated_cost"].mean(),
                "mean_trace_quality": group["trace_quality"].mean(),
            }
        )

    return pd.DataFrame(rows)


summary = summarize_condition(results)
summary

#### **Intervalos de confianza bootstrap**

Los intervalos de confianza permiten distinguir variación observada de una diferencia estable.

Se usa bootstrap no paramétrico sobre ejecuciones.

In [ ]:
def bootstrap_mean_interval(
    values: np.ndarray,
    samples: int,
    seed: int,
    confidence: float = 0.95,
) -> tuple[float, float, float]:
    """Calcula media e intervalo bootstrap."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)

    bootstrap_means = np.empty(samples)

    for index in range(samples):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )
        bootstrap_means[index] = sample.mean()

    alpha = 1.0 - confidence

    lower = float(
        np.quantile(
            bootstrap_means,
            alpha / 2.0,
        )
    )
    upper = float(
        np.quantile(
            bootstrap_means,
            1.0 - alpha / 2.0,
        )
    )

    return float(values.mean()), lower, upper

In [ ]:
interval_records = []

metric_names = [
    "task_success",
    "attack_success",
    "unauthorized_action",
    "persistent_contamination",
    "trace_quality",
]

for (
    attack_present,
    defense_enabled,
), group in results.groupby(
    [
        "attack_present",
        "defense_enabled",
    ]
):
    for metric_name in metric_names:
        mean_value, lower, upper = bootstrap_mean_interval(
            values=group[metric_name].to_numpy(),
            samples=BOOTSTRAP_SAMPLES,
            seed=(
                MASTER_SEED
                + int(attack_present) * 10
                + int(defense_enabled) * 20
                + len(metric_name)
            ),
        )

        interval_records.append(
            {
                "attack_present": attack_present,
                "defense_enabled": defense_enabled,
                "metric": metric_name,
                "mean": mean_value,
                "ci_lower": lower,
                "ci_upper": upper,
            }
        )

intervals = pd.DataFrame(interval_records)
intervals.head()

#### **Efecto pareado de la defensa**

Se compara la misma combinación de escenario, modalidad y semilla con y sin defensa.

Esto reduce variación causada por diferencias entre casos.

In [ ]:
paired = (
    results.pivot_table(
        index=[
            "scenario_id",
            "modality",
            "attack_type",
            "attack_present",
            "seed",
        ],
        columns="defense_enabled",
        values=[
            "task_success",
            "attack_success",
            "unauthorized_action",
            "trace_quality",
        ],
    )
    .reset_index()
)

paired.columns = [
    "_".join(
        str(part)
        for part in column
        if str(part) != ""
    )
    if isinstance(column, tuple)
    else str(column)
    for column in paired.columns
]

paired["delta_task_success"] = (
    paired["task_success_True"]
    - paired["task_success_False"]
)

paired["delta_attack_success"] = (
    paired["attack_success_True"]
    - paired["attack_success_False"]
)

paired["delta_unauthorized_action"] = (
    paired["unauthorized_action_True"]
    - paired["unauthorized_action_False"]
)

paired["delta_trace_quality"] = (
    paired["trace_quality_True"]
    - paired["trace_quality_False"]
)

paired.groupby(
    "attack_present",
)[
    [
        "delta_task_success",
        "delta_attack_success",
        "delta_unauthorized_action",
        "delta_trace_quality",
    ]
].mean()

#### **Análisis por modalidad**

Una media global puede ocultar ataques difíciles en una modalidad concreta.

Se reporta utilidad y seguridad por modalidad.

In [ ]:
attacked_results = results[
    results["attack_present"]
]

modality_summary = (
    attacked_results.groupby(
        [
            "modality",
            "defense_enabled",
        ],
        as_index=False,
    )
    .agg(
        task_success_rate=("task_success", "mean"),
        attack_success_rate=("attack_success", "mean"),
        unauthorized_action_rate=(
            "unauthorized_action",
            "mean",
        ),
        mean_trace_quality=("trace_quality", "mean"),
    )
)

modality_summary

In [ ]:
plot_data = (
    modality_summary.pivot(
        index="modality",
        columns="defense_enabled",
        values="attack_success_rate",
    )
    .rename(
        columns={
            False: "sin_defensa",
            True: "con_defensa",
        }
    )
)

ax = plot_data.plot(
    kind="bar",
    figsize=(8, 4),
)

ax.set_title("Tasa de éxito del ataque por modalidad")
ax.set_xlabel("Modalidad")
ax.set_ylabel("Tasa de éxito del ataque")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y")
plt.xticks(rotation=0)
plt.show()

#### **Frontera utilidad y seguridad**

Una defensa útil reduce ataques y conserva éxito de tarea.

El análisis no debe optimizar una sola métrica.

In [ ]:
benign_summary = (
    results[
        ~results["attack_present"]
    ]
    .groupby(
        "defense_enabled",
        as_index=False,
    )
    .agg(
        benign_utility=("task_success", "mean"),
        over_refusal=("over_refusal", "mean"),
    )
)

attack_summary = (
    results[
        results["attack_present"]
    ]
    .groupby(
        "defense_enabled",
        as_index=False,
    )
    .agg(
        attack_success_rate=("attack_success", "mean"),
        unauthorized_action_rate=(
            "unauthorized_action",
            "mean",
        ),
    )
)

tradeoff = benign_summary.merge(
    attack_summary,
    on="defense_enabled",
)

tradeoff["security_gain"] = (
    1.0 - tradeoff["attack_success_rate"]
)

tradeoff["balanced_score"] = (
    0.5 * tradeoff["benign_utility"]
    + 0.5 * tradeoff["security_gain"]
)

tradeoff

In [ ]:
ax = tradeoff.plot(
    x="attack_success_rate",
    y="benign_utility",
    kind="scatter",
    s=120,
    figsize=(6, 4),
)

for _, row in tradeoff.iterrows():
    label = (
        "con defensa"
        if row["defense_enabled"]
        else "sin defensa"
    )
    ax.annotate(
        label,
        (
            row["attack_success_rate"],
            row["benign_utility"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

ax.set_title("Utilidad benigna frente a éxito del ataque")
ax.set_xlabel("Tasa de éxito del ataque")
ax.set_ylabel("Utilidad benigna")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.grid(True)
plt.show()

#### **Criterio de decisión**

Una defensa se considera favorable cuando cumple tres condiciones.

1. Reduce la tasa de éxito del ataque

2. Reduce acciones no autorizadas

3. Mantiene una utilidad benigna aceptable.

In [ ]:
without_defense = tradeoff[
    ~tradeoff["defense_enabled"]
].iloc[0]

with_defense = tradeoff[
    tradeoff["defense_enabled"]
].iloc[0]

decision_report = {
    "attack_reduction": float(
        without_defense["attack_success_rate"]
        - with_defense["attack_success_rate"]
    ),
    "utility_change": float(
        with_defense["benign_utility"]
        - without_defense["benign_utility"]
    ),
    "unauthorized_action_reduction": float(
        without_defense["unauthorized_action_rate"]
        - with_defense["unauthorized_action_rate"]
    ),
    "defense_is_favorable": bool(
        with_defense["attack_success_rate"]
        < without_defense["attack_success_rate"]
        and with_defense["benign_utility"] >= 0.80
        and with_defense["unauthorized_action_rate"]
        < without_defense["unauthorized_action_rate"]
    ),
}

decision_report

#### **Lectura de resultados**

La defensa debe interpretarse mediante utilidad, seguridad, costo y trazabilidad.

Una reducción de ataques no es suficiente si el sistema rechaza demasiadas tareas legítimas.

Una utilidad alta no es suficiente si el agente conserva autoridad para ejecutar acciones no autorizadas.

El análisis por modalidad permite detectar superficies donde la defensa sigue siendo débil.

#### **Amenazas a la validez**

El agente y los ataques son simulados.

Las probabilidades representan un laboratorio controlado y no un modelo desplegado.

Las tareas son pequeñas y las herramientas no acceden a sistemas externos.

El bootstrap estima variación interna del benchmark y no validez externa.

El protocolo debe repetirse con modelos reales antes de extraer conclusiones operativas.

#### **Preguntas de desarrollo**

1. ¿Por qué una defensa puede mejorar seguridad y reducir utilidad al mismo tiempo?

2. ¿Qué ventaja ofrece una comparación pareada?

3. ¿Por qué la tasa de éxito del ataque debe calcularse solo sobre casos atacados?

4. ¿Qué oculta una media global por encima de todas las modalidades?

5. ¿Cómo se interpreta un intervalo de confianza que se superpone entre condiciones?

6. ¿Qué métrica impediría aprobar una defensa con mucho sobrerrechazo?.

#### **Exportación de resultados**

El cuaderno guarda ejecuciones, resúmenes, intervalos, análisis pareado y metadatos.

In [ ]:
results.to_csv(
    RESULTS_DIR / "benchmark_runs.csv",
    index=False,
)

summary.to_csv(
    RESULTS_DIR / "benchmark_summary.csv",
    index=False,
)

intervals.to_csv(
    RESULTS_DIR / "bootstrap_intervals.csv",
    index=False,
)

paired.to_csv(
    RESULTS_DIR / "paired_effects.csv",
    index=False,
)

modality_summary.to_csv(
    RESULTS_DIR / "modality_summary.csv",
    index=False,
)

tradeoff.to_csv(
    RESULTS_DIR / "utility_security_tradeoff.csv",
    index=False,
)

with (RESULTS_DIR / "decision_report.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_report,
        file,
        indent=2,
        ensure_ascii=False,
    )

metadata = {
    "curso": "MCC225",
    "semana": 13,
    "cuaderno": "Cuaderno32-MCC225",
    "tema": "Evaluación integral de utilidad y seguridad en agentes",
    "semilla_principal": MASTER_SEED,
    "semillas_de_ejecucion": RUN_SEEDS,
    "muestras_bootstrap": BOOTSTRAP_SAMPLES,
    "numero_de_escenarios": len(SCENARIOS),
    "numero_de_ejecuciones": len(results),
    "modo": "CPU sin APIs externas",
    "alcance": "Benchmark sintético y defensivo",
}

with (RESULTS_DIR / "metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Resultados exportados en:", RESULTS_DIR)

#### **Conclusión**

Una defensa agéntica debe evaluarse como un compromiso entre utilidad y seguridad.

El protocolo de cuatro condiciones separa el costo de la defensa de su eficacia frente a ataques.

Las semillas múltiples, los intervalos bootstrap, las comparaciones pareadas y el análisis por modalidad convierten una demostración aislada en evidencia experimental reproducible.